# Chamada de variantes (*Variant Calling*)

Adaptado de [Data Carpentry — Wrangling Genomics, Episódio 4](https://datacarpentry.github.io/wrangling-genomics/04-variant_calling.html).

As células deste notebook rodam em **Bash** (kernel `bash_kernel`), então os comandos são idênticos aos que você digitaria no terminal.

---

## Objetivos

Ao final desta lição você será capaz de:

- **Inspecionar** um arquivo de alinhamento **BAM** (`samtools view`, `samtools flagstat`)
- **Indexar** um BAM (`samtools index`)
- **Chamar variantes** com `bcftools mpileup` e `bcftools call`
- **Filtrar** as variantes chamadas (`vcfutils.pl varFilter`)
- **Visualizar** as variantes no alinhamento, no terminal (`samtools tview`) e no **IGV-Web**

## Configuração inicial

Rode a célula abaixo **uma vez** para garantir que estamos na raiz do repositório (onde ficam as pastas `data/` e `results/`). A partir daí, todos os caminhos serão escritos como no terminal do Data Carpentry.

In [ ]:
# Garante que o diretório de trabalho é a raiz do repositório
# (onde ficam as pastas data/ e results/).
# Rode esta célula uma vez, no início. É seguro rodá-la mais de uma vez.
[ -d data ] || cd ..
pwd
echo "--- data/ (dados de entrada) ---"; ls data
echo "--- results/ (saídas) ---";        ls results

## Seção 0 — Preparação (já executada) ⚙️

> **Você não precisa rodar esta seção.** Os comandos abaixo já foram executados para preparar o material, e os resultados já estão no repositório. Estão aqui **apenas para reprodutibilidade** — para mostrar de onde veio o arquivo BAM com que a aula começa.

Esta aula assume que você **já viu o mapeamento de *reads* (*read mapping*)**. Por isso, começamos a partir de um alinhamento pronto. As etapas de preparação foram:

1. **Download** do genoma de referência de *E. coli* e dos *reads* já filtrados (*trimmed*).
2. **Indexação** do genoma de referência para o `bwa` (`bwa index`).
3. **Alinhamento** dos *reads* ao genoma com `bwa mem`, gerando um arquivo **SAM**.
4. **Conversão** SAM → BAM (`samtools view`) e **ordenação** por posição (`samtools sort`).

O resultado é o arquivo `results/bam/SRR2584866.aligned.sorted.bam`, o ponto de partida da nossa aula.

In [ ]:
# ⚙️ PREPARAÇÃO — já executado; NÃO é necessário rodar em aula.
# Mantido comentado apenas para reprodutibilidade (de onde veio o BAM inicial).
#
# --- 1. Dados (baixados uma vez) ---
# Genoma de referência: Escherichia coli B str. REL606 (NC_012967.1 / CP000819.1),
# 4.629.812 bp. Obtido do dataset oficial do Data Carpentry no figshare
# ("Data Carpentry Genomics beta 2.0", backup.tar.gz):
#   https://figshare.com/articles/dataset/Data_Carpentry_Genomics_beta_2_0/7726454
# -> data/ref_genome/ecoli_rel606.fasta
#
# ATENÇÃO ao problema detectado:
#   (a) O link do NCBI que aparece na página do episódio
#       (GCA_000017985.1_ASM1798v1_genomic.fna.gz) baixa um arquivo corrompido.
#   (b) NÃO use o genoma de E. coli K-12 MG1655 (ASM584v2, 4.641.652 bp): os reads
#       são da linhagem B/REL606, e mapeá-los contra K-12 produz ~31.000 variantes
#       (divergência ENTRE linhagens) em vez das ~800 esperadas.
#
# Reads já filtrados (subconjunto do Data Carpentry, via figshare):
#   https://ndownloader.figshare.com/files/14418248  ->  data/trimmed_fastq_small/
#
# --- 2. Indexar a referência para o bwa ---
# bwa index data/ref_genome/ecoli_rel606.fasta
#
# --- 3. Alinhar os reads (bwa mem) -> SAM ---
# mkdir -p results/sam results/bam
# bwa mem data/ref_genome/ecoli_rel606.fasta \
#   data/trimmed_fastq_small/SRR2584866_1.trim.sub.fastq \
#   data/trimmed_fastq_small/SRR2584866_2.trim.sub.fastq \
#   > results/sam/SRR2584866.aligned.sam
#
# --- 4. SAM -> BAM (samtools view) e ordenar por posição (samtools sort) ---
# samtools view -S -b results/sam/SRR2584866.aligned.sam \
#   | samtools sort -o results/bam/SRR2584866.aligned.sorted.bam -

---

## Seção 1 — Inspecionar o alinhamento

Nossa aula começa aqui. Temos um alinhamento pronto e **ordenado por posição**:
`results/bam/SRR2584866.aligned.sorted.bam`.

O formato **BAM** é a versão **binária e comprimida** do formato **SAM** (texto). Por ser binário, não conseguimos lê-lo diretamente com `head` ou `cat` — usamos `samtools view` para traduzi-lo de volta para texto SAM legível.

Vamos espiar as primeiras linhas de alinhamento:

In [ ]:
# Onde está o samtools? (confirma que estamos usando o do ambiente conda)
which samtools

# As primeiras linhas de alinhamento (samtools traduz o BAM binário de volta para texto SAM)
samtools view results/bam/SRR2584866.aligned.sorted.bam | head

### Estatísticas gerais do alinhamento

O comando `samtools flagstat` dá um resumo rápido do alinhamento: total de *reads*, quantos foram mapeados, quantos formam pares corretamente alinhados, etc. É uma boa checagem de sanidade **antes** de chamar variantes.

In [ ]:
# Resumo do alinhamento: total de reads, mapeados, pares corretos, etc.
samtools flagstat results/bam/SRR2584866.aligned.sorted.bam

---

## Seção 2 — Indexar o BAM

Assim como um índice no fim de um livro permite pular direto para uma página, o **índice de um BAM** (`.bai`) permite que ferramentas acessem rapidamente os *reads* de uma **região específica** do genoma, sem varrer o arquivo inteiro.

O BAM **precisa estar ordenado por posição** para ser indexado — o nosso já está (foi ordenado na preparação). O índice será necessário mais adiante, na etapa de **visualização** (`samtools tview`).

O comando gera um arquivo `.bai` ao lado do `.bam`:

In [ ]:
# Cria o índice (arquivo .bai ao lado do .bam)
samtools index results/bam/SRR2584866.aligned.sorted.bam

# Confirma que o índice foi criado
ls -lh results/bam/

---

## Seção 3 — Chamar variantes (*variant calling*)

Aqui está o núcleo da lição. A chamada de variantes com o `bcftools` tem **dois passos**:

**Passo 1 — `bcftools mpileup`:** percorre o genoma e, para cada posição, resume o que os *reads* alinhados mostram ali — profundidade de cobertura, bases observadas e suas qualidades. O resultado é um arquivo **BCF** (a versão binária do VCF) contendo as *verossimilhanças dos genótipos* em cada posição.

Passamos a **referência** com `-f`, para o `bcftools` saber qual é a base esperada em cada posição, e `-O b` para a saída sair em BCF binário. Primeiro criamos as pastas de saída:

In [ ]:
# Onde está o bcftools? (confirma que estamos usando o do ambiente conda)
which bcftools

# Pastas para os resultados (ignoradas pelo git — são saídas que você gera)
mkdir -p results/bcf results/vcf

# Passo 1: resume, posição a posição, o que os reads mostram (-> BCF)
bcftools mpileup -O b \
  -o results/bcf/SRR2584866_raw.bcf \
  -f data/ref_genome/ecoli_rel606.fasta \
  results/bam/SRR2584866.aligned.sorted.bam

**Passo 2 — `bcftools call`:** a partir das verossimilhanças do passo anterior, decide onde há de fato uma variante e escreve um arquivo **VCF**.

Três opções importantes:

- `--ploidy 1` — *E. coli* é **haploide** (um único cromossomo), então dizemos ao `bcftools` para **não** assumir genótipos diploides.
- `-m` — usa o modelo de chamada **multialélico** (recomendado).
- `-v` — reporta **apenas os sítios variantes** (*variant-only*), ignorando as posições idênticas à referência.

In [ ]:
# Passo 2: chama as variantes a partir do BCF (-> VCF)
bcftools call --ploidy 1 -m -v \
  -o results/vcf/SRR2584866_variants.vcf \
  results/bcf/SRR2584866_raw.bcf

### Espiando o VCF

O arquivo **VCF** (*Variant Call Format*) tem três partes: um **cabeçalho** (linhas iniciadas por `##`) que descreve como o arquivo foi gerado; uma **linha de nomes de colunas** (iniciada por um único `#`, com `#CHROM POS ID REF ALT ...`); e, em seguida, **uma linha por variante**.

Vamos ver as primeiras variantes, pulando o cabeçalho `##`:

In [ ]:
# Mostra a linha de colunas (#CHROM ...) e as primeiras variantes, sem o cabeçalho ##
grep -v "^##" results/vcf/SRR2584866_variants.vcf | head

### Entendendo as colunas do VCF

Cada linha de variante tem **8 colunas fixas**, seguidas de informações de genótipo. Tomando a primeira variante como exemplo:

```
NC_012967.1  1521  .  C  T  207.417  .  DP=9;...;MQ=60  GT:PL:AD  1:237,0:0,9
```

| # | Coluna | Valor no exemplo | Significado |
|---|--------|------------------|-------------|
| 1 | `CHROM`  | `NC_012967.1` | cromossomo / contig da referência |
| 2 | `POS`    | `1521` | posição na referência (contada a partir de 1) |
| 3 | `ID`     | `.` | identificador da variante (`.` = nenhum) |
| 4 | `REF`    | `C` | base na **referência** |
| 5 | `ALT`    | `T` | base **alternativa** encontrada nos *reads* |
| 6 | `QUAL`   | `207.417` | confiança na chamada, em escala Phred (**quanto maior, melhor**) |
| 7 | `FILTER` | `.` | status de filtragem (`.` = ainda não filtrado; `PASS` = passou) |
| 8 | `INFO`   | `DP=9;…;MQ=60` | anotações do sítio (ver abaixo) |

Depois das 8 colunas fixas vêm as de **genótipo**:

- `FORMAT` = `GT:PL:AD` — lista quais campos são reportados, na ordem;
- a última coluna (`1:237,0:0,9`) traz os **valores** desses campos para a nossa amostra.

**Alguns campos úteis:**

| Campo | Onde | Significado |
|-------|------|-------------|
| `DP`  | INFO / FORMAT | **profundidade**: nº de *reads* cobrindo a posição (aqui, 9) |
| `MQ`  | INFO | qualidade média de **mapeamento** dos *reads* (60 = ótima) |
| `GT`  | FORMAT | **genótipo**: `1` = alelo alternativo. É um único número porque *E. coli* é **haploide** (num diploide seria algo como `0/1` ou `1/1`) |
| `AD`  | FORMAT | *reads* que suportam cada alelo: `0,9` = 0 para `REF`, 9 para `ALT` |

> 💡 **Repare:** na posição 1521, `DP=9` e `AD=0,9` — **todos os 9 *reads* mostram o alelo alternativo**, nenhum mostra a referência. É uma variante bem sustentada. Guarde essa ideia: veremos a variante vizinha (posição 1612) *de verdade*, olhando os *reads*, na Seção 5.

---

## Seção 4 — Filtrar as variantes

As variantes que acabamos de chamar são **brutas**: entre elas há chamadas em que não confiamos muito — posições com pouquíssimos *reads*, com qualidade baixa, ou aglomeradas em torno de *indels* (onde o alinhamento é notoriamente instável).

O `vcfutils.pl` (que vem junto com o `bcftools`) traz o `varFilter`, que aplica um conjunto de **filtros padrão** e descarta essas chamadas menos confiáveis. Entre os critérios: profundidade mínima de *reads*, qualidade mínima de mapeamento, e remoção de variantes muito próximas de *indels*.

> ⚠️ **Não espere uma redução dramática.** Filtragem aqui é uma **limpeza**, não uma peneira fina: como nossa cobertura é razoável e *E. coli* é haploide, a maioria das chamadas já é confiável e sobrevive ao filtro. Poucas dezenas serão removidas — e isso é o esperado (o próprio Data Carpentry reporta 686 → 675).

In [ ]:
# Onde está o vcfutils.pl? (repare: fica no mesmo diretório do bcftools —
# ele vem junto com o pacote)
which vcfutils.pl

# Aplica os filtros padrão do varFilter, gerando o VCF "final"
vcfutils.pl varFilter results/vcf/SRR2584866_variants.vcf \
  > results/vcf/SRR2584866_final_variants.vcf

### Quanto o filtro removeu?

Vamos comparar o número de variantes antes e depois:

In [ ]:
# Quantas variantes antes e depois do filtro?
echo "antes do filtro:  $(grep -vc '^#' results/vcf/SRR2584866_variants.vcf)"
echo "depois do filtro: $(grep -vc '^#' results/vcf/SRR2584866_final_variants.vcf)"

### Na prática: filtragem mais rigorosa

O `varFilter` é um filtro **conservador** — ele descarta o pior, não faz uma seleção rigorosa. Numa análise real, você normalmente aplicaria critérios explícitos e justificáveis, por exemplo com `bcftools filter`:

```bash
bcftools filter -i 'QUAL>=30 && DP>=10' results/vcf/SRR2584866_variants.vcf
```

Isso manteria apenas variantes com **qualidade** ≥ 30 e **profundidade** ≥ 10. Os limiares corretos dependem da cobertura, do organismo e da pergunta biológica — não existe um valor universal.

> 💡 **Pense a respeito:** por que filtrar por profundidade (`DP`) importa tanto? Uma variante vista em 2 *reads* e uma vista em 50 *reads* têm o mesmo grau de confiança?

---

## Seção 5 — Visualizar as variantes

Números num arquivo VCF são abstratos. **Olhar** para o alinhamento é o que constrói intuição sobre o que é uma variante confiável — e é assim que se investiga uma chamada suspeita na prática.

O `samtools tview` mostra os *reads* empilhados sobre a referência, direto no terminal. Normalmente ele é **interativo** (você navega com as setas do teclado), o que não funciona dentro de uma célula de notebook. Por isso usamos duas opções que o tornam **não-interativo**:

- `-d T` — imprime a saída como **texto puro**, em vez de abrir a interface navegável.
- `-p` — pula direto para uma **posição** específica do genoma.

Vamos olhar a variante da posição **1612** (`A` → `G`), uma das que sobreviveram à filtragem. Note que isto só funciona porque **indexamos o BAM** na Seção 2:

In [ ]:
# Visualiza o alinhamento na posição 1612, onde há uma variante A -> G.
#   -d T  : saída em texto puro (não-interativa), adequada para o notebook
#   -p    : pula direto para a posição indicada
samtools tview -d T -p NC_012967.1:1612 \
  results/bam/SRR2584866.aligned.sorted.bam \
  data/ref_genome/ecoli_rel606.fasta | head -20

### Como ler a saída do `tview`

A saída tem três partes, de cima para baixo:

1. Uma **régua** com as coordenadas do genoma.
2. A sequência da **referência**.
3. Os ***reads*** alinhados, um por linha.

Nos *reads*, a notação é:

| Símbolo | Significado |
|---------|-------------|
| `.` | base **igual** à referência, *read* na fita **direta** (*forward*) |
| `,` | base **igual** à referência, *read* na fita **reversa** |
| `A C G T` (maiúsculas) | base **diferente** da referência, fita direta |
| `a c g t` (minúsculas) | base **diferente** da referência, fita reversa |
| espaço em branco | região não coberta por aquele *read* |

Ou seja: **um mar de pontos e vírgulas com uma coluna de letras** é exatamente a assinatura visual de uma variante. Na posição 1612, a referência tem `A`, mas **todos os *reads* mostram `G`** — em ambas as fitas. Essa concordância entre fitas é um forte indício de que a variante é **real**, e não um artefato de sequenciamento.

> 🔍 **Exercício:** escolha outra posição da lista de variantes filtradas e visualize-a. Alguma parece menos convincente que a da posição 1612? O que a tornaria suspeita — poucos *reads*? Discordância entre as fitas? Bases variantes só em uma das pontas dos *reads*?

---

## Seção 6 — Extra: visualizar no IGV-Web 🔬

O `tview` é prático, mas cru. O **IGV** (*Integrative Genomics Viewer*) é a ferramenta padrão da área para explorar dados genômicos visualmente.

Vamos usar o **IGV-Web**, que roda **inteiramente no navegador**: nada para instalar, em nenhum sistema operacional.

### 👉 [https://igv.org/app/](https://igv.org/app/)

### Passo 1 — Carregar o genoma de referência

O REL606 **não** está na lista de genomas prontos do IGV, então vamos carregá-lo manualmente.

No menu **`Genome`** → **`Local File...`**, selecione **os dois arquivos juntos**:

- `data/ref_genome/ecoli_rel606.fasta`
- `data/ref_genome/ecoli_rel606.fasta.fai`

> ⚠️ O `.fai` (o índice) é **obrigatório**. Sem ele o IGV não consegue navegar pelo FASTA. Selecione os dois arquivos na mesma caixa de diálogo.

### Passo 2 — Carregar as variantes

No menu **`Tracks`** → **`Local File...`**, selecione:

- `results/vcf/SRR2584866_final_variants.vcf`

Cada variante aparecerá como uma marca na trilha. Clicando numa delas, o IGV mostra os campos do VCF (posição, alelos `REF`/`ALT`, qualidade, profundidade).

### Passo 3 — Navegar até uma variante

Na caixa de busca, digite a mesma posição que vimos com o `tview`:

```
NC_012967.1:1612
```

> 💡 Se estiver rodando no **Binder**, os arquivos estão no servidor, não no seu computador. Baixe-os primeiro: no JupyterLab, botão direito no arquivo → **Download**. O VCF final tem só 110 KB e a referência 4,5 MB.

### Opcional — ver os *reads* também

Para reproduzir no IGV o que o `tview` mostrou, carregue também o alinhamento (`Tracks` → `Local File...`), selecionando **os dois arquivos juntos**:

- `results/bam/SRR2584866.aligned.sorted.bam`
- `results/bam/SRR2584866.aligned.sorted.bam.bai`

Navegando até `NC_012967.1:1612`, você verá a mesma coluna de bases discordantes da Seção 5 — agora colorida e navegável.

> 🔍 **Compare:** o que fica mais fácil de perceber no IGV do que no `tview`? E o que o `tview` tem de vantagem? (Dica: pense em rodar isso remotamente, por SSH, num servidor sem interface gráfica.)